# Module 06: Named Entity Recognition


# 6.1 NER Fundamentals


## 🏷️ What is Named Entity Recognition?

Named Entity Recognition (NER) is the process of locating and classifying "named entities" in unstructured text into predefined categories such as person names, organizations, locations, medical codes, time expressions, quantities, monetary values, percentages, etc.

In spaCy, entities are accessed via the `doc.ents` property. This returns a tuple of `Span` objects.


In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")
text = "Tim Cook is the CEO of Apple. He lives in California."
doc = nlp(text)

print("Found Entities:")
for ent in doc.ents:
    print(f"- Text: {ent.text:<15} | Label: {ent.label_}")


Found Entities:
- Text: Tim Cook        | Label: PERSON
- Text: Apple           | Label: ORG
- Text: California      | Label: GPE


## ⚠️ Handling Missing Entities

Statistical models are not perfect. They rely on context to make predictions. If the context is ambiguous or the capitalization is unusual, the model might miss an entity.

Always remember that NER models are probabilistic!


In [2]:
# Notice how removing capitalization completely breaks the NER model's predictions
text_lower = "tim cook is the ceo of apple. he lives in california."
doc_lower = nlp(text_lower)

print("Entities found in lowercase text:")
for ent in doc_lower.ents:
    print(f"- {ent.text} ({ent.label_})")
print(f"Total entities found: {len(doc_lower.ents)}")


Entities found in lowercase text:
- tim cook (PERSON)
- california (GPE)
Total entities found: 2



<br><br>

---

<br><br>


# 6.2 Built-in Entity Types


## 📚 The OntoNotes 5 Corpus

The default English models in spaCy (`en_core_web_sm`, `md`, `lg`) are trained on the **OntoNotes 5** dataset. This dataset defines 18 standard entity types.

Let's write a quick script to find an example of several different entity types from a complex text!


In [1]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

# A sentence packed with different entity types
complex_text = """
On March 15th, 2024, John Smith, the VP of Microsoft, traveled to Paris, France. 
He bought a masterpiece painting for $1.5 million. He speaks fluent French and 
his flight took 8 hours. Nearly 45% of the company's revenue comes from Europe.
"""

doc = nlp(complex_text)

data = []
for ent in doc.ents:
    data.append({
        "Entity Text": ent.text,
        "Label": ent.label_,
        "Description": spacy.explain(ent.label_)
    })

pd.DataFrame(data)


,Entity Text,Label,Description
0,"March 15th, 2024",DATE,Absolute or relative dates or periods
1,John Smith,PERSON,"People, including fictional"
2,Microsoft,ORG,"Companies, agencies, institutions, etc."
3,Paris,GPE,"Countries, cities, states"
4,France,GPE,"Countries, cities, states"
5,$1.5 million,MONEY,"Monetary values, including unit"
6,French,NORP,Nationalities or religious or political groups
7,8 hours,TIME,Times smaller than a day
8,Nearly 45%,PERCENT,"Percentage, including ""%"""
9,Europe,LOC,"Non-GPE locations, mountain ranges, bodies of ..."


### Standard Entity Reference Guide

- **PERSON**: People, including fictional.
- **NORP**: Nationalities or religious or political groups.
- **FAC**: Buildings, airports, highways, bridges, etc.
- **ORG**: Companies, agencies, institutions, etc.
- **GPE**: Countries, cities, states.
- **LOC**: Non-GPE locations, mountain ranges, bodies of water.
- **PRODUCT**: Objects, vehicles, foods, etc. (Not services).
- **EVENT**: Named hurricanes, battles, wars, sports events, etc.
- **WORK_OF_ART**: Titles of books, songs, etc.
- **LAW**: Named documents made into laws.
- **LANGUAGE**: Any named language.
- **DATE**: Absolute or relative dates or periods.
- **TIME**: Times smaller than a day.
- **PERCENT**: Percentage, including "%" .
- **MONEY**: Monetary values, including unit.
- **QUANTITY**: Measurements, as of weight or distance.
- **ORDINAL**: "first", "second", etc.
- **CARDINAL**: Numerals that do not fall under another type.



<br><br>

---

<br><br>


# 6.3 Entity Properties


## 🔍 Deep Dive into Entity Spans

Because an entity is just a `Span` object under the hood, it has access to all the same properties we covered in Module 2.

However, entities also have specific properties related to Knowledge Base (KB) linking.


In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp("Elon Musk founded SpaceX in 2002.")
ent = doc.ents[0] # 'Elon Musk'

print(f"Text: {ent.text}")
print(f"Start token index: {ent.start}")
print(f"End token index: {ent.end}")
print(f"Root token: {ent.root}")


Text: Elon Musk
Start token index: 0
End token index: 2
Root token: Musk


## 🔗 Knowledge Base (KB) IDs and Entity Linking

Sometimes, knowing something is a `PERSON` is not enough. Which "John Smith" are we talking about?

**Entity Linking** (or Named Entity Disambiguation) is the process of linking an entity to a unique identifier in a Knowledge Base (like Wikidata or Wikipedia). 

By default, the standard models don't perform entity linking (it requires a specialized pipeline component), but the slots for these IDs exist on the entity object!


In [2]:
print(f"Knowledge Base ID: {ent.kb_id_}")
print("(This is empty because our current pipeline does not include an Entity Linker component)")


Knowledge Base ID: 
(This is empty because our current pipeline does not include an Entity Linker component)


## 🧩 Filtering by Entity Type

A very common NLP task is to extract all entities of a specific type (e.g., getting a list of all organizations mentioned in a news article).


In [3]:
article = "Google, Microsoft, and Amazon are tech giants. New York is a big city."
doc = nlp(article)

# Using list comprehension to filter
organizations = [ent.text for ent in doc.ents if ent.label_ == "ORG"]

print(f"Organizations found: {organizations}")


Organizations found: ['Google', 'Microsoft', 'Amazon']



<br><br>

---

<br><br>


# 6.4 NER and Context


## 🧠 The Importance of Context

Unlike regular expressions, spaCy's NER models do not look for specific words. They look at the *context* surrounding the words. 

A word like "Apple" can be a fruit or a company. A word like "Washington" can be a person, a state, or a city. The NER model uses the surrounding tokens to disambiguate the meaning.


In [2]:
import spacy
nlp = spacy.load("en_core_web_sm")

# Context 1: Company
doc1 = nlp("Apple announced a new iPhone today.")
print(f"Doc 1 'Apple': {doc1.ents[0].label_}")

# Context 2: Fruit (Notice there are no entities here!)
doc2 = nlp("I ate a delicious apple for breakfast.")
print(f"Doc 2 Entities: {doc2.ents}")

# Context 3: Person
doc3 = nlp("George Washington was the first president.")
print(f"Doc 3 'Washington': {doc3.ents[0].label_}")

# Context 4: Location (GPE)
doc4 = nlp("I am traveling to Washington next week.")
print(f"Doc 4 'Washington': {doc4.ents[0].label_}")


Doc 1 'Apple': ORG
Doc 2 Entities: ()
Doc 3 'Washington': PERSON
Doc 4 'Washington': GPE


## 🕵️‍♂️ Coreference Resolution (Brief Intro)

A common issue with NER is pronouns. 

"Bill Gates founded Microsoft. **He** is a billionaire."

We know "He" refers to Bill Gates, but the NER model only flags "Bill Gates" and "Microsoft". Resolving "He" back to "Bill Gates" is called **Coreference Resolution**.

Historically, spaCy handled this via an extension called `neuralcoref`. In modern spaCy (3.x+), coreference resolution is available as an experimental pipeline component via the `spacy-experimental` package.

*If you need to extract all facts about a person, you must often combine NER with Coreference Resolution!*



<br><br>

---

<br><br>


# 6.5 Visualizing Entities


## 🎨 displaCy for Entities

As we saw in Module 5, `displaCy` is fantastic for highlighting entities in text. Let's look at advanced formatting for it!


In [1]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

text = "When Sebastian Thrun started working on self-driving cars at Google in 2007, few people outside of the company took him seriously."
doc = nlp(text)

# Basic render
displacy.render(doc, style="ent", jupyter=True)


## 🖌️ Custom Styling

You can pass an `options` dictionary to displaCy to change exactly how entities are displayed. This is especially useful when you start training your own custom entity types (like `MEDICAL_CONDITION` or `PART_NUMBER`) and want them to stand out!


In [2]:
# Let's style PERSON and ORG with custom colors
colors = {
    "PERSON": "linear-gradient(90deg, #aa9cfc, #fc9ce7)",
    "ORG": "#09a3d5",
    "DATE": "#bdecb6"
}

options = {
    "ents": ["PERSON", "ORG", "DATE"], # Only show these entities
    "colors": colors
}

displacy.render(doc, style="ent", jupyter=True, options=options)


## 🖥️ Rendering Outside Jupyter

If you are writing a standard Python script (not a notebook), you can generate raw HTML to save to a file, or use the built-in web server!


In [6]:
# Generate raw HTML
html = displacy.render(doc, style="ent", page=True, options=options)

# You could then save it:
# with open("entities.html", "w") as f:
#     f.write(html)

print("HTML generated successfully! (First 200 characters):")
print(html[:200] + "...")


HTML generated successfully! (First 200 characters):


TypeError: 'NoneType' object is not subscriptable


<br><br>

---

<br><br>


# 6.6 Evaluating NER Performance


## 📊 How do we measure NER accuracy?

Because NER involves both finding the correct boundary (the start and end of the word) AND classifying it correctly (is it an ORG or a PERSON?), we don't just use standard "accuracy".

We use three metrics:
1. **Precision (P)**: Out of all the entities the model *guessed*, how many were actually correct?
2. **Recall (R)**: Out of all the *actual* entities in the text, how many did the model find?
3. **F1-Score (F)**: The harmonic mean of Precision and Recall. This is the main metric used to judge NER models.

### Why do we need both?
If a model guesses that *every single word* is a PERSON, its Recall will be 100% (it found all the people!), but its Precision will be near 0% (almost all of its guesses were wrong).

If a model makes only 1 guess and is correct, its Precision is 100%, but its Recall is terrible (it missed everything else).


## 🛠️ The Scorer Object

spaCy provides a `Scorer` object to evaluate pipelines. You provide it with `Example` objects (which contain the model's prediction alongside the "Gold Standard" correct answers).

*(Note: We will dive much deeper into `Example` objects and evaluation in Part 5: Training. For now, understand that NER models are evaluated purely on P, R, and F-score).*


In [1]:
import spacy
from spacy.training import Example
from spacy.scorer import Scorer

nlp = spacy.load("en_core_web_sm")

# Let's mock up an evaluation
text = "Mark Zuckerberg founded Facebook."
predicted_doc = nlp(text)

# Create a reference doc with the correct answers (Gold Standard)
reference_doc = nlp.make_doc(text)

# Let's pretend the correct answers are Mark Zuckerberg (PERSON) and Facebook (ORG)
from spacy.tokens import Span
reference_doc.ents = [
    Span(reference_doc, 0, 2, label="PERSON"),
    Span(reference_doc, 3, 4, label="ORG")
]

# Create an Example object linking the prediction to the reference
example = Example(predicted_doc, reference_doc)

# Score the pipeline
scorer = Scorer()
scores = scorer.score([example])

print("=== NER SCORES ===")
print(f"Precision: {scores['ents_p']}")
print(f"Recall:    {scores['ents_r']}")
print(f"F1-Score:  {scores['ents_f']}")


=== NER SCORES ===
Precision: 1.0
Recall:    0.5
F1-Score:  0.6666666666666666


## 🎉 Summary of Module 6

You now have a solid grasp on Named Entity Recognition!
- You learned the 18 standard OntoNotes 5 entity types.
- You saw how context completely changes model predictions.
- You learned how to extract specific entity types, look at their KB IDs, and visualize them beautifully with displaCy.
- You understand the basics of P/R/F1 scoring.

In **Module 7**, we will move to **Word Vectors & Similarity**, unlocking the ability to compare the semantic meaning of different words and documents!
